In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Basic libraries
# IMPORT LIBRARIES
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

import warnings
warnings.filterwarnings("ignore")


In [ ]:
# Load train and test data
train = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

print("Training Data Shape:", train.shape)
print("Testing Data Shape:", test.shape)

In [ ]:
#Exploratory Data Analysis

# Distribution of SalePrice
plt.figure(figsize=(8,5))
sns.histplot(train['SalePrice'], kde=True, color='skyblue')
plt.title('Distribution of Sale Prices')
plt.show()

# Correlation heatmap (numeric features)
plt.figure(figsize=(12,8))
corr = train.corr(numeric_only=True)
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title("Feature Correlations")
plt.show()


In [ ]:
#Data Cleanning and Preprocessing

# Combine train and test for consistent preprocessing
full = pd.concat([train.drop('SalePrice', axis=1), test], axis=0)

# Fill missing numeric columns with median, categorical with mode
for col in full.columns:
    if full[col].dtype == "object":
        full[col] = full[col].fillna(full[col].mode()[0])
    else:
        full[col] = full[col].fillna(full[col].median())

# One-hot encode categorical variables
full = pd.get_dummies(full, drop_first=True)

# Split back into train and test sets
X = full.iloc[:len(train), :]
X_test = full.iloc[len(train):, :]
y = train['SalePrice']

# Log-transform target for stability
y_log = np.log1p(y)

# Split train/validation set
X_train, X_valid, y_train, y_valid = train_test_split(X, y_log, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)



In [ ]:
#Experiment 1: Baseline linear

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=10),
    "Lasso Regression": Lasso(alpha=0.001)
}

results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_valid_scaled)
    rmse = np.sqrt(mean_squared_error(y_valid, preds))
    r2 = r2_score(y_valid, preds)
    results[name] = {"RMSE": rmse, "R2": r2}
    print(f"{name}: RMSE = {rmse:.3f}, R² = {r2:.3f}")

In [ ]:
#Experiment 2: Feature Engineering + Regularized Regression (Ridge & Lasso)
# Use top 20 correlated features with SalePrice
corr_matrix = train.corr(numeric_only=True)
top_features = corr_matrix["SalePrice"].sort_values(ascending=False).head(20).index
X_top = X[top_features.drop("SalePrice", errors="ignore")]

# Split and scale
X_train2, X_valid2, y_train2, y_valid2 = train_test_split(X_top, y_log, test_size=0.2, random_state=42)
scaler2 = StandardScaler()
X_train2_scaled = scaler2.fit_transform(X_train2)
X_valid2_scaled = scaler2.transform(X_valid2)

# Ridge regression
ridge = Ridge(alpha=10)
ridge.fit(X_train2_scaled, y_train2)
preds2 = ridge.predict(X_valid2_scaled)
rmse2 = np.sqrt(mean_squared_error(y_valid2, preds2))
r2_2 = r2_score(y_valid2, preds2)
print(f"\nExperiment 2 (Ridge on top features): RMSE = {rmse2:.3f}, R² = {r2_2:.3f}")

In [ ]:
#Experiment 3:Polynomial Regression + Feature Engineering
top10_features = corr_matrix["SalePrice"].sort_values(ascending=False).head(10).index
X_poly = X[top10_features.drop("SalePrice", errors="ignore")]

# Split and scale
X_train3, X_valid3, y_train3, y_valid3 = train_test_split(X_poly, y_log, test_size=0.2, random_state=42)
scaler3 = StandardScaler()
X_train3_scaled = scaler3.fit_transform(X_train3)
X_valid3_scaled = scaler3.transform(X_valid3)

# Polynomial Features (degree 2)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train3_scaled)
X_valid_poly = poly.transform(X_valid3_scaled)

# Ridge regression on polynomial features
ridge_poly = Ridge(alpha=5)
ridge_poly.fit(X_train_poly, y_train3)
preds3 = ridge_poly.predict(X_valid_poly)
rmse3 = np.sqrt(mean_squared_error(y_valid3, preds3))
r2_3 = r2_score(y_valid3, preds3)
print(f"\nExperiment 3 (Polynomial Ridge): RMSE = {rmse3:.3f}, R² = {r2_3:.3f}")

In [ ]:
#Model Comparison

results_df = pd.DataFrame({
    "Experiment 1": results["Ridge Regression"]["RMSE"],
    "Experiment 2": rmse2,
    "Experiment 3": rmse3
}, index=["RMSE"])
print("\nModel Comparison (RMSE):\n")
print(results_df.T)

# Optional: Visual comparison
results_df.T.plot(kind="bar", figsize=(8,5), legend=False)
plt.title("Regression Model Comparison (Lower RMSE = Better)")
plt.ylabel("RMSE")
plt.show()

# Visual comparison
results_df.T.plot(kind="bar", figsize=(8,5), legend=False)
plt.title("Regression Model Comparison (Lower RMSE = Better)")
plt.ylabel("RMSE")
plt.show()


In [ ]:
#Final Model & Prediction
# Use the best-performing model (Ridge or Lasso typically)
X_train_final = X_train_poly
X_test_final = poly.transform(scaler3.transform(X_test[top10_features.drop("SalePrice", errors="ignore")]))

ridge_poly.fit(X_train_final, y_train3)
final_preds = ridge_poly.predict(X_test_final)
final_preds = np.expm1(final_preds)  # Reverse log-transform

# Submission file
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": final_preds
})
submission.to_csv("submission_experiment3.csv", index=False)
print("✅ Submission file created: submission_experiment3.csv")